# Real-Time EEG-to-Text Training Architecture

This notebook implements the real-time EEG-to-Text models based on state-of-the-art Voice-to-Text ASR techniques.

## Models Included:
1. **Conformer (Convolution-augmented Transformer)**: The gold standard for ASR. It uses 1D spatial convolutions for feature extraction, followed by Conformer blocks to capture both local micro-states (CNN) and global sequence intent (Transformer).
2. **CNN-LSTM (DeepSpeech style)**: A classical approach for sequential signal processing.

**Loss Function**: Connectionist Temporal Classification (CTC) for alignment-free training.

In [ ]:
# Requirements: pip install torch torchaudio h5py matplotlib
import glob
import os

import h5py
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn, optim
from torch.utils.data import Dataset

DATASET_PATH = r"C:\SDE Projects\Open-BCI-EEG-Waves-To-Text-Translation-And-further-Robotic-Implementaions\dataset\extracted"
MODELS_DIR = "models"
METRICS_DIR = "metrics"

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)

## 1. Dataset Loading

In [ ]:
class EEGDataset(Dataset):
    def __init__(self, data_dir, max_len=500):
        self.files = glob.glob(os.path.join(data_dir, "**", "*.h5"), recursive=True)
        self.max_len = max_len
        # Simple vocabulary character-level mapping
        self.chars = (
            "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789 .,!?'-"
        )
        self.char_to_idx = {ch: i + 1 for i, ch in enumerate(self.chars)}
        self.char_to_idx["<PAD>"] = 0
        self.vocab_size = len(self.char_to_idx)

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file_path = self.files[idx]
        filename = os.path.basename(file_path)
        transcript = filename.replace(".h5", "").replace("_", " ")

        with h5py.File(file_path, "r") as f:
            keys = list(f.keys())
            if len(keys) > 0:
                eeg_data = f[keys[0]][:]  # shape expected: [105 channels, time_steps]
            else:
                eeg_data = np.zeros((105, self.max_len))

        if len(eeg_data.shape) != 2 or eeg_data.shape[0] != 105:
            eeg_data = np.zeros((105, self.max_len))
        if eeg_data.shape[1] < self.max_len:
            pad_width = self.max_len - eeg_data.shape[1]
            eeg_data = np.pad(eeg_data, ((0, 0), (0, pad_width)), mode="constant")
        else:
            eeg_data = eeg_data[:, : self.max_len]

        # [time_steps, channels]
        eeg_tensor = torch.tensor(eeg_data, dtype=torch.float32).transpose(0, 1)

        target = [self.char_to_idx.get(c, 0) for c in transcript]
        target_tensor = torch.tensor(target, dtype=torch.long)

        subject = (
            os.path.normpath(file_path).split(os.sep)[-3]
            if len(os.path.normpath(file_path).split(os.sep)) >= 3
            else "Unknown"
        )
        return eeg_tensor, target_tensor, subject


dataset = EEGDataset(DATASET_PATH)
print(f"Found {len(dataset)} HDF5 files for training.")

## 2. Models Definition
### 2.1 Conformer (Convolution-augmented Transformer)
State-of-the-art ASR model adapted for EEG spatial decoding.

In [ ]:
from torchaudio.models import Conformer


class ConformerEEG(nn.Module):
    def __init__(self, num_channels=105, input_dim=256, num_classes=65):
        super().__init__()
        # Spatial Convolution to map 105 channels to internal representation
        self.spatial_conv = nn.Conv1d(num_channels, input_dim, kernel_size=3, padding=1)

        # Torchaudio Conformer expects inputs of (batch, time, input_dim)
        self.conformer = Conformer(
            input_dim=input_dim,
            num_heads=4,
            ffn_dim=512,
            num_layers=4,
            depthwise_conv_kernel_size=31,
        )

        # Linear head for character prediction (CTC)
        self.fc = nn.Linear(input_dim, num_classes)

    def forward(self, x):
        # x shape: [batch, time, channels]
        x = x.transpose(1, 2)  # [batch, channels, time]
        x = self.spatial_conv(x)  # [batch, input_dim, time]
        x = x.transpose(1, 2)  # [batch, time, input_dim]

        # Conformer requires input lengths, using max length for simplicity
        lengths = torch.full((x.size(0),), x.size(1), dtype=torch.long, device=x.device)

        out, _ = self.conformer(x, lengths)
        out = self.fc(out)  # [batch, time, num_classes]
        return out

### 2.2 DeepSpeech-style (CNN-LSTM)

In [ ]:
class DeepSpeechEEG(nn.Module):
    def __init__(self, num_channels=105, hidden_size=256, num_classes=65):
        super().__init__()
        self.conv1 = nn.Conv1d(num_channels, 64, kernel_size=5, stride=1, padding=2)
        self.bn1 = nn.BatchNorm1d(64)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(2)

        self.lstm = nn.LSTM(
            64, hidden_size, num_layers=2, batch_first=True, bidirectional=True
        )
        self.fc = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.pool(self.relu(self.bn1(self.conv1(x))))
        x = x.transpose(1, 2)

        self.lstm.flatten_parameters()
        out, _ = self.lstm(x)
        out = self.fc(out)
        return out

## 3. Training Loop (CTC Loss)

In [ ]:
import difflib

from IPython.display import display
from tqdm.auto import tqdm


def greedy_decoder(logits, char_to_idx):
    idx_to_char = {v: k for k, v in char_to_idx.items()}
    # logits shape: [time, batch, classes]
    preds = torch.argmax(logits, dim=-1).transpose(0, 1)  # [batch, time]
    decoded = []
    for seq in preds:
        prev = -1
        text = ""
        for c in seq:
            c = c.item()
            if c != 0 and c != prev:
                text += idx_to_char.get(c, "")
            prev = c
        decoded.append(text)
    return decoded


def calculate_cer(pred_texts, target_texts):
    scores = []
    for p, t in zip(pred_texts, target_texts):
        # ratio() returns 1.0 for perfect match, approximate for CER/Recall
        scores.append(difflib.SequenceMatcher(None, p, t).ratio())
    return sum(scores) / max(1, len(scores))


def train_model(model, name, dataloader, epochs=5, char_to_idx=None):
    criterion = nn.CTCLoss(blank=0, zero_infinity=True)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    loss_history = []
    recall_history = []
    model.train()

    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))
    display_handle = display(fig, display_id=True)
    plt.close(fig)  # Prevent duplicate static rendering

    for epoch in range(epochs):
        epoch_loss = 0
        epoch_recall = 0

        pbar = tqdm(dataloader, desc=f"[{name}] Epoch {epoch + 1}/{epochs}")
        for eeg, target, subject in pbar:
            batch_size = eeg.size(0)
            subj = subject[0] if len(subject) > 0 else "Unknown"

            # Forward pass
            optimizer.zero_grad()
            out = model(eeg)
            out = out.transpose(0, 1)  # [time, batch, classes]
            out_log_sm = nn.functional.log_softmax(out, dim=2)

            out_time = (
                eeg.size(1) // 2
                if "DeepSpeech" in type(model).__name__
                else eeg.size(1)
            )
            input_lengths = torch.full(
                size=(batch_size,), fill_value=out_time, dtype=torch.long
            )
            target_lengths = torch.tensor(
                [len(t) if isinstance(t, list) else len(t[0]) for t in target]
            )
            # Handle potential nested lists from dataloader for targets
            target_flat = []
            for t in target:
                if isinstance(t, list):
                    target_flat.extend(t)
                else:
                    target_flat.extend(
                        t[0].tolist() if isinstance(t[0], torch.Tensor) else t[0]
                    )

            # Compute loss
            loss = criterion(out_log_sm, target, input_lengths, target_lengths)
            loss.backward()
            optimizer.step()

            batch_loss = loss.item()
            epoch_loss += batch_loss

            # Compute Metrics (Recall/Accuracy)
            batch_recall = 0
            if char_to_idx is not None:
                preds = greedy_decoder(out_log_sm, char_to_idx)
                # target back to text
                idx_to_char = {v: k for k, v in char_to_idx.items()}
                targets_text = []
                for t in target:
                    t_list = t if isinstance(t, list) else t[0].tolist()
                    targets_text.append(
                        "".join([idx_to_char.get(c, "") for c in t_list])
                    )
                batch_recall = calculate_cer(preds, targets_text)
            epoch_recall += batch_recall

            # Update Live Visuals
            ax1.clear()
            ax2.clear()
            ax3.clear()

            sample = eeg[0].detach().cpu().numpy()
            for ch in range(min(5, sample.shape[1])):
                ax1.plot(sample[:, ch] + ch * 2.0, label=f"Ch {ch}")
            ax1.legend(loc="upper right")
            ax1.set_title(f"Live Raw EEG (Subj: {subj})")
            ax1.set_xlabel("Time")

            ax2.plot(loss_history + [batch_loss], color="r", label="CTC Loss")
            ax2.set_title("Real-Time CTC Loss")
            ax2.set_xlabel("Batches")
            ax2.legend(loc="upper right")

            ax3.plot(
                recall_history + [batch_recall], color="g", label="Recall (Accuracy)"
            )
            ax3.set_title("Real-Time Character Recall")
            ax3.set_xlabel("Batches")
            ax3.legend(loc="upper right")

            display_handle.update(fig)
            pbar.set_postfix(
                {"Loss": f"{batch_loss:.4f}", "Recall": f"{batch_recall:.4f}"}
            )

        avg_loss = epoch_loss / max(1, len(dataloader))
        avg_recall = epoch_recall / max(1, len(dataloader))
        loss_history.append(avg_loss)
        recall_history.append(avg_recall)

    import os

    if not os.path.exists("models"):
        os.makedirs("models")
    torch.save(model.state_dict(), os.path.join("models", f"{name}_checkpoint.pth"))
    return loss_history, recall_history